In [ ]:
import json
import csv
import os
import signal
from datetime import datetime
from kafka import KafkaConsumer

KAFKA_BROKER = "localhost:9092"
TOPIC = "energy-sensors"
OUTPUT_FILE = "training_data.csv"

FIELDS = [
    "sensor_id", "plant_id", "timestamp",
    "power_consumption", "voltage", "current",
    "temperature", "power_factor", "frequency",
    "is_anomaly"
]

running = True

def handle_stop(sig, frame):
    global running
    print("\nStopping consumer, saving file...")
    running = False

signal.signal(signal.SIGINT, handle_stop)

def main():
    consumer = KafkaConsumer(
        TOPIC,
        bootstrap_servers=KAFKA_BROKER,
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        auto_offset_reset="earliest",
        group_id="energysentinel-collector",
    )

    file_exists = os.path.exists(OUTPUT_FILE)

    with open(OUTPUT_FILE, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)

        if not file_exists:
            writer.writeheader()

        print(f"EnergySentinel Consumer started → reading from topic: {TOPIC}")
        print(f"Saving to: {OUTPUT_FILE}")
        print("Press Ctrl+C to stop and save\n")

        count = 0
        anomaly_count = 0

        while running:
            records = consumer.poll(timeout_ms=1000)
            for _, messages in records.items():
                for msg in messages:
                    reading = msg.value
                    writer.writerow({field: reading.get(field, "") for field in FIELDS})
                    count += 1
                    if reading.get("is_anomaly") == 1:
                        anomaly_count += 1

                    if count % 100 == 0:
                        f.flush()
                        print(f"Collected: {count} readings | Anomalies: {anomaly_count} ({anomaly_count/count*100:.1f}%)")

    consumer.close()
    print(f"\nDone! Saved {count} readings to {OUTPUT_FILE}")
    print(f"Anomalies: {anomaly_count} ({anomaly_count/count*100:.1f}% of total)" if count > 0 else "")

if __name__ == "__main__":
    main()